In [ ]:
import pandas as pd
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen
from urllib.error import URLError, HTTPError

In [ ]:
### Set the timeframe for the files we want to retrieve
start_year = 2023
end_year = 2026

## Specify the file you want to extract from each zip
## fs220.txt contains the financial data for credit unions
file_names = ['fs220.txt', 'FS220.txt', 'fs220A.txt', 'FS220A.txt']

## Base URL pattern for recent NCUA quarterly data (2015+)
base_url = 'https://www.ncua.gov/files/publications/analysis/call-report-data-'

In [ ]:
## Build the list of quarter dates to fetch
dates = []
for year in range(start_year, end_year):
    for quarter in ['03', '06', '09', '12']:
        dates.append(f'{year}-{quarter}')

## Set up SSL context (macOS Python often needs this)
import ssl
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

from urllib.request import Request

def download_zip(date_str):
    """Download and return a ZipFile for the given quarter date string."""
    url = base_url + date_str + '.zip'
    req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    resp = urlopen(req, context=ssl_ctx)
    return url, ZipFile(BytesIO(resp.read()))

def open_file_from_zip(zf, name_options):
    """Try to open a file from the zip using multiple possible names."""
    for name in name_options:
        if name in zf.namelist():
            return zf.open(name)
    raise FileNotFoundError(f"None of {name_options} found in zip. Available: {zf.namelist()}")

## Fetch and process each quarter
df = pd.DataFrame()
successful = []
failed = []

for date_str in dates:
    try:
        url, zf = download_zip(date_str)
        print(f"OK: {url}")

        ## Extract account descriptions (maps column codes to readable names)
        acct_desc_file = open_file_from_zip(zf, ['AcctDesc.txt', 'Acct_Desc.txt', 'Acct_Des.txt'])
        acct_desc_df = pd.read_csv(acct_desc_file, encoding='ISO-8859-1')

        ## Extract the financial data
        fs220_file = open_file_from_zip(zf, file_names)
        fs220_df = pd.read_csv(fs220_file, encoding='ISO-8859-1')
        print(f"    {date_str}: {fs220_df.shape[0]} credit unions, {fs220_df.shape[1]} columns")

        ## Map column codes to descriptive names
        column_names = pd.DataFrame({'fs220': fs220_df.columns.str.lower()})
        acct_desc_df['Account'] = acct_desc_df['Account'].str.lower()
        new_columns = pd.merge(column_names, acct_desc_df, how='left', left_on='fs220', right_on='Account')
        new_columns['AcctName'] = new_columns['AcctName'].fillna(new_columns['fs220'])
        fs220_df.columns = fs220_df.columns + " - " + new_columns['AcctName']

        df = pd.concat([df, fs220_df], axis=0, sort=False)
        successful.append(date_str)

    except (HTTPError, URLError) as e:
        print(f"SKIP: {date_str} - not available ({e})")
        failed.append(date_str)
    except Exception as e:
        print(f"ERROR: {date_str} - {type(e).__name__}: {e}")
        failed.append(date_str)

print(f"\nDone! Retrieved {len(successful)}/{len(dates)} quarters.")
print(f"Total rows: {len(df):,}, Total columns: {df.shape[1]}")
if failed:
    print(f"Missing quarters: {failed}")

In [ ]:
## Preview the data
df.head()

In [ ]:
## Export to CSV
output_file = 'NCUA_Call_Report.csv'
df.to_csv(output_file, index=False)
print(f"Saved {len(df):,} rows to {output_file}")